# 动手练习：网络可控性与稳定性

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。数据文件放在与本笔记本同级的 `data/` 文件夹中。

## 任务 1：Kalman 秩判据

线性系统 $\dot{\mathbf x}=A\mathbf x+B\mathbf u$ 中，$A_{ij}\neq0$ 表示有向边 $j\to i$，$B$ 的每一列对应一个驱动节点。可控性矩阵
$\mathcal C=[B,\;AB,\;A^2B,\;\dots,\;A^{N-1}B]$ 满秩（$\operatorname{rank}\mathcal C=N$）时系统可控。

1. 补全函数 `ctrb_rank(A, B)`，返回 $\mathcal C$ 的秩；
2. 用 setup 里的随机非零权重，分别计算：链 $0\to1\to2\to3$ 只驱动节点 0 的秩 `r_chain`；星 $0\to1,0\to2,0\to3$ 只驱动节点 0 的秩 `r_star`；星形再加驱动节点 2、3 的秩 `r_star3`。

期望看到：链可控（秩 4）；星只驱动中心时不可控（秩 2，三个叶子只能收到同一个信号）；星加上两个叶子后可控。

> **提示：** `np.hstack([np.linalg.matrix_power(A, k) @ B for k in range(n)])`。

In [ ]:
# 准备（直接运行）
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


def system_matrix(edges, n, seed=0):
    """边 u->v 写在 A[v, u]，权重取 0.5–1.5 之间的随机数。"""
    rng = np.random.default_rng(seed)
    A = np.zeros((n, n))
    for u, v in edges:
        A[v, u] = rng.uniform(0.5, 1.5)
    return A


def input_matrix(drivers, n):
    B = np.zeros((n, len(drivers)))
    for j, d in enumerate(drivers):
        B[d, j] = 1.0
    return B


chain = [(0, 1), (1, 2), (2, 3)]
star = [(0, 1), (0, 2), (0, 3)]

In [ ]:
def ctrb_rank(A, B):
    n = A.shape[0]
    # TODO: 拼出 C = [B, AB, ..., A^{n-1} B]，返回 np.linalg.matrix_rank(C)
    return None


r_chain = None   # TODO: 链，只驱动节点 0
r_star = None    # TODO: 星，只驱动节点 0
r_star3 = None   # TODO: 星，驱动节点 0、2、3

In [ ]:
# 核对（不用修改）
assert ctrb_rank(np.eye(2), np.ones((2, 1))) is not None, "ctrb_rank 还返回 None，请补全函数"
assert ctrb_rank(np.array([[0.0, 0.0], [1.0, 0.0]]), np.array([[1.0], [0.0]])) == 2, "两节点链 0→1 驱动节点 0 应满秩 2；检查 A 的幂次是否从 0 取到 n−1"
assert ctrb_rank(np.zeros((3, 3)), np.eye(3)[:, :1]) == 1, "A=0 时只有 B 本身一列有效，秩应为 1"
assert r_chain == 4, "链 0→1→2→3 只驱动节点 0 应可控（秩 4）；检查是否用了 system_matrix(chain, 4)"
assert r_star == 2, "星形只驱动中心时秩应为 2：B 一个方向，AB 一个方向，A²B = 0"
assert r_star3 == 4, "星形再驱动叶子 2、3 后应可控（秩 4）；检查 input_matrix([0, 2, 3], 4)"
print("✓ 任务 1 通过")

## 任务 2：用最大匹配找驱动节点

最小输入定理：$N_D=\max(N-|M^*|,1)$。计算方法：把有向图拆成二分图，每个节点 $u$ 有出端 `("o", u)` 和入端 `("i", u)`，有向边 $u\to v$ 连接 `("o", u)`–`("i", v)`。最大匹配后，**入端没有被匹配**的节点就是驱动节点；若全部匹配（完美匹配），任选 1 个节点。

1. 补全 `driver_nodes(G)`，返回驱动节点列表；
2. 对 setup 中的随机有向网络 `G`（12 个节点、18 条边），求驱动节点 `D`；
3. 用任务 1 的 Kalman 判据验证：只驱动 `D` 中的节点，随机权重下可控性矩阵的秩 `rank_D` 应等于 12。

期望看到：$N_D$ 等于 $N$ 减去匹配边数；`rank_D = 12`。

> **提示：** `nx.bipartite.hopcroft_karp_matching` 返回的字典对每条匹配边存了两次（两个方向），所以匹配边数是 `len(M) // 2`；判断入端是否被匹配用 `("i", v) in M`。

In [ ]:
# 准备（直接运行）
G = nx.gnm_random_graph(12, 18, directed=True, seed=4)
A_G = system_matrix(G.edges(), 12, seed=1)
A_G /= max(1.0, np.abs(np.linalg.eigvals(A_G)).max())   # 缩放，避免高次幂数值溢出

In [ ]:
def driver_nodes(G):
    Bg = nx.Graph()
    top = [("o", u) for u in G]
    Bg.add_nodes_from(top)
    Bg.add_nodes_from([("i", v) for v in G])
    Bg.add_edges_from((("o", u), ("i", v)) for u, v in G.edges())
    M = nx.bipartite.hopcroft_karp_matching(Bg, top_nodes=top)
    # TODO: 返回入端 ("i", v) 不在 M 中的节点 v 组成的列表；若为空，返回 [任一节点]
    return None


D = None        # TODO: driver_nodes(G)
rank_D = None   # TODO: ctrb_rank(A_G, input_matrix(D, 12))

In [ ]:
# 核对（不用修改）
assert D is not None and rank_D is not None, "D、rank_D 还没有赋值"
assert driver_nodes(nx.DiGraph(chain)) == [0], "链 0→1→2→3 的驱动节点应只有 [0]"
assert len(driver_nodes(nx.DiGraph(star))) == 3, "星形 0→1,2,3 应有 3 个驱动节点（中心只能匹配一个叶子，另外两个叶子和中心本身未匹配）"
assert len(driver_nodes(nx.cycle_graph(5, create_using=nx.DiGraph))) == 1, "有向环存在完美匹配，N_D 应为 1（不能返回空列表）"
_B = nx.Graph()
_B.add_nodes_from([("o", u) for u in G] + [("i", v) for v in G])
_B.add_edges_from((("o", u), ("i", v)) for u, v in G.edges())
_m = len(nx.bipartite.hopcroft_karp_matching(_B, top_nodes=[("o", u) for u in G])) // 2
assert len(D) == max(12 - _m, 1), "N_D 应等于 N − |M*| = %d；检查是否只数了入端未匹配的节点" % max(12 - _m, 1)
assert set(D) <= set(G.nodes()), "D 里应是 G 的节点编号，而不是 ('i', v) 这样的元组"
assert rank_D == 12, "只驱动 D 时 Kalman 秩应为 12；检查 input_matrix(D, 12) 和 ctrb_rank"
print("✓ 任务 2 通过")

## 任务 3：控制的鲁棒性：哪些边是关键边

删除一条边后，若所需驱动节点数 $N_D$ **增加**，这条边称为**关键边**（critical edge，Liu 等 2011）。网络删掉一条边后可能仍然连通，却需要更多输入——连通鲁棒性和控制鲁棒性是两回事。

对 setup 中的随机有向网络 `H`（20 个节点、30 条边）：

1. 计算原网络的驱动节点数 `nd0`；
2. 逐条删除每条边（在副本上删），找出全部关键边，存为列表 `critical`（元素是 `(u, v)` 元组）；
3. 计算关键边占全部边的比例 `frac_critical`。

期望看到：只有一部分边是关键边；删除非关键边时 $N_D$ 不变（删边不会让 $N_D$ 减少）。

> **提示：** setup 提供了参考实现 `driver_nodes_ref`，后面的任务也会用到它。`H2 = H.copy(); H2.remove_edge(u, v)`。

In [ ]:
# 准备（直接运行）
def driver_nodes_ref(G):
    Bg = nx.Graph()
    top = [("o", u) for u in G]
    Bg.add_nodes_from(top)
    Bg.add_nodes_from([("i", v) for v in G])
    Bg.add_edges_from((("o", u), ("i", v)) for u, v in G.edges())
    M = nx.bipartite.hopcroft_karp_matching(Bg, top_nodes=top)
    D = [v for v in G if ("i", v) not in M]
    return D if D else [next(iter(G))]


H = nx.gnm_random_graph(20, 30, directed=True, seed=7)

In [ ]:
nd0 = None            # TODO: 原网络的 N_D
critical = None       # TODO: 删除后 N_D 增加的边 (u, v) 列表
frac_critical = None  # TODO: len(critical) / 边数

In [ ]:
# 核对（不用修改）
assert nd0 is not None and critical is not None and frac_critical is not None, "nd0、critical、frac_critical 还没有赋值"
assert nd0 == len(driver_nodes_ref(H)), "nd0 应为原网络的驱动节点数"
_ref = set()
for _u, _v in H.edges():
    _H2 = H.copy()
    _H2.remove_edge(_u, _v)
    if len(driver_nodes_ref(_H2)) > nd0:
        _ref.add((_u, _v))
_got = set(map(tuple, critical))
assert _got <= set(H.edges()), "critical 里应是 H 的边 (u, v)；检查方向是否写反"
assert _got == _ref, "关键边应有 %d 条，你找到 %d 条；检查是否在副本上删边、是否用“>”而不是“!=”比较" % (len(_ref), len(_got))
assert abs(frac_critical - len(_ref) / H.number_of_edges()) < 1e-9, "frac_critical = 关键边数 / 总边数"
print("✓ 任务 3 通过")

## 任务 4：驱动节点比例与平均度、度异质性

Liu 等（2011）发现：驱动节点比例 $n_D=N_D/N$ 主要由度分布决定；网络越稀疏、度越异质，越难控制；驱动节点倾向于避开枢纽。

对 $N=1000$ 的有向 ER 网络和有向无标度网络（setup 中的静态模型，$\gamma=2.5$），在边数 $L=cN$（$c=1,2,3,4,6$，即平均**出度** $c$，每个节点的入度＋出度平均为 $2c$）下：

1. 计算两类网络的 $n_D$，按 $c$ 的顺序存入列表 `nd_er`、`nd_sf`；
2. 在 $c=3$ 的无标度网络 `G_sf3` 上，计算全网节点的平均总度（入度＋出度）`deg_all` 和驱动节点的平均总度 `deg_driver`。

期望看到：$c$ 越大 $n_D$ 越小；同一 $c$ 下无标度网络的 $n_D$ 更高；驱动节点的平均度低于全网平均度。

> **提示：** `G.degree()` 在有向图上给出入度＋出度。

In [ ]:
# 准备（直接运行）
def sf_directed(N, c, gamma=2.5, seed=0):
    """静态模型：节点权重 w_i ∝ i^{-1/(gamma-1)}，按权重分别抽取起点和终点，共 L = c N 条边（无自环、无重边）。"""
    rng = np.random.default_rng(seed)
    w = np.arange(1, N + 1) ** (-1.0 / (gamma - 1))
    w /= w.sum()
    L = int(c * N)
    codes = np.array([], dtype=np.int64)
    while codes.size < L:
        u = rng.choice(N, 4 * L, p=w)
        v = rng.choice(N, 4 * L, p=w)
        codes = np.concatenate([codes, (u * N + v)[u != v]])
        _, idx = np.unique(codes, return_index=True)
        codes = codes[np.sort(idx)]
    G = nx.DiGraph()
    G.add_nodes_from(range(N))
    G.add_edges_from((int(x // N), int(x % N)) for x in codes[:L])
    return G


N = 1000
cs = [1, 2, 3, 4, 6]
nets_er = [nx.gnm_random_graph(N, c * N, directed=True, seed=c) for c in cs]
nets_sf = [sf_directed(N, c, seed=c) for c in cs]
G_sf3 = nets_sf[cs.index(3)]

In [ ]:
nd_er = []   # TODO: 对 nets_er 中每个网络追加 len(driver_nodes_ref(G)) / N
nd_sf = []   # TODO: 对 nets_sf 同样计算
deg_all = None      # TODO: G_sf3 全网平均总度
deg_driver = None   # TODO: G_sf3 驱动节点的平均总度

In [ ]:
# 核对（不用修改）
assert len(nd_er) == 5 and len(nd_sf) == 5, "nd_er、nd_sf 应各有 5 个值，对应 c = 1, 2, 3, 4, 6"
assert all(0 < x <= 1 for x in nd_er + nd_sf), "n_D 应在 (0, 1] 之间；检查是否除以了 N"
assert all(nd_er[i] > nd_er[i + 1] for i in range(4)), "ER 网络的 n_D 应随平均度增大而下降"
assert all(s > e for s, e in zip(nd_sf, nd_er)), "同一平均度下无标度网络的 n_D 应高于 ER 网络；检查两个列表是否写反"
assert abs(nd_er[0] - len(driver_nodes_ref(nets_er[0])) / N) < 1e-9, "nd_er[0] 应为 c = 1 的 ER 网络的 N_D / N"
assert deg_all is not None and deg_driver is not None, "deg_all、deg_driver 还没有赋值"
assert abs(deg_all - 2 * G_sf3.number_of_edges() / N) < 1e-6, "全网平均总度应为 2L/N = 6；检查是否用了 G_sf3.degree()"
assert deg_driver < deg_all, "驱动节点的平均度应低于全网平均度（驱动节点避开枢纽）"
print("✓ 任务 4 通过")

## 任务 5：随机失效与蓄意攻击

对 BA 网络（$N=1000$，$m=2$）和边数相同的 ER 网络，逐步删除比例 $f$ 的节点：随机删除，或按**初始**度从高到低删除（攻击）。记录删除后最大连通分支占原网络 $N$ 的比例。

1. 补全 `giant_curve(G, order)`：对 `fs` 中每个 $f$，删去 `order` 中前 `int(f * N)` 个节点，返回最大分支比例的列表；
2. 用它得到四条曲线，存入字典 `curves`，键为 `("ER", "random")`、`("ER", "attack")`、`("BA", "random")`、`("BA", "attack")`；随机顺序用 setup 中的 `rand_order`；
3. 用 $f_c=1-1/(\kappa_0-1)$、$\kappa_0=\langle k^2\rangle/\langle k\rangle$ 估计两个网络的随机失效阈值，存入字典 `fc = {"ER": …, "BA": …}`。

期望看到：BA 网络的 $\kappa_0$ 和理论 $f_c$ 更高；按度攻击时 BA 网络比 ER 网络碎得更快。

> **提示：** `max((len(c) for c in nx.connected_components(H)), default=0) / N`。

In [ ]:
# 准备（直接运行）
N = 1000
G_ba = nx.barabasi_albert_graph(N, 2, seed=1)
G_er = nx.gnm_random_graph(N, G_ba.number_of_edges(), seed=1)
nets = {"ER": G_er, "BA": G_ba}
fs = np.linspace(0, 0.9, 19)
rand_order = list(np.random.default_rng(0).permutation(N))

In [ ]:
def giant_curve(G, order):
    out = []
    for f in fs:
        # TODO: 在 G 的副本上删去 order[: int(f * N)]，记录最大连通分支 / N
        pass
    return out


curves = {}   # TODO: 四条曲线
fc = {}       # TODO: {"ER": ..., "BA": ...}

In [ ]:
# 核对（不用修改）
_keys = {("ER", "random"), ("ER", "attack"), ("BA", "random"), ("BA", "attack")}
assert set(curves) == _keys, "curves 应有四个键：(ER/BA, random/attack)"
assert all(len(y) == len(fs) for y in curves.values()), "每条曲线的长度应等于 len(fs) = 19；检查 giant_curve 是否 append 了结果"
assert all(abs(y[0] - max(len(c) for c in nx.connected_components(nets[n])) / N) < 1e-9 for (n, _), y in curves.items()), "f = 0 时应为原网络最大分支比例；检查是否除以了原网络的 N"
_i = int(np.argmin(np.abs(fs - 0.1)))
assert curves[("BA", "attack")][_i] < curves[("BA", "random")][_i], "删去 10% 节点时，BA 网络在攻击下的最大分支应小于随机失效下的"
assert curves[("BA", "attack")][_i] < curves[("ER", "attack")][_i], "按度攻击 10% 节点后，BA 网络应比 ER 网络碎得更厉害"
for _n, _G in nets.items():
    _k = np.array([d for _, d in _G.degree()], dtype=float)
    _f = 1 - 1 / ((_k**2).mean() / _k.mean() - 1)
    assert abs(fc[_n] - _f) < 1e-3, "%s 的 f_c 应约为 %.3f；kappa0 = <k²>/<k>，注意 <k²> 是度平方的平均，不是平均度的平方" % (_n, _f)
assert fc["BA"] > fc["ER"], "BA 网络度更异质，理论 f_c 应高于 ER 网络"
print("✓ 任务 5 通过")

## 任务 6：小分析：人类转录调控网络有多难控制

用真实数据 `data/collectri_human_tf_target_ccby_curated.csv`（CollecTRI 人类 TF→靶基因调控关系中有 CC BY 4.0 人工整理来源支持的子集；出处见本专题的真实数据案例和 `data/SOURCES.md`）。setup 已把它读成整数编号的有向边数组 `u → v`（去掉了重复边和自环），并提供快速计算驱动节点数的函数 `n_driver(u, v, N)`（用 SciPy 的二分图最大匹配，结果与 Hopcroft–Karp 相同）。

自己设计并完成：

1. 真实网络的驱动节点比例 `nd_real`；
2. 保度随机化：打乱靶基因列 `v`（每个 TF 的出度、每个基因的入度保持不变），去掉由此产生的自环和重复边，重复 `n_rand` 次（自己定，至少 3 次），把每次的 $n_D$ 存入列表 `nd_rand`；
3. 源节点（没有入边）占驱动节点的比例 `frac_source`。

然后想一想：$n_D$ 和保度随机网络接近吗？这支持“度分布决定可控性”吗？驱动节点主要来自哪里——与控制轮廓中“转录调控网络以外部扩张为主”的说法对得上吗？这些驱动节点能直接当作药物靶点吗？（见专题页对“结构上需要的驱动节点 ≠ 可操作的干预靶点”的讨论。）

In [ ]:
# 准备（直接运行）
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

_df = pd.read_csv("data/collectri_human_tf_target_ccby_curated.csv").drop_duplicates(["tf", "target"])
_df = _df[_df.tf != _df.target]
genes = sorted(set(_df.tf) | set(_df.target))
_idx = {g: i for i, g in enumerate(genes)}
N_real = len(genes)
u = _df.tf.map(_idx).to_numpy()
v = _df.target.map(_idx).to_numpy()


def n_driver(u, v, N):
    """N_D = max(N − |M*|, 1)，二分图：出端 u（行）— 入端 v（列）。"""
    M = csr_matrix((np.ones(len(u)), (u, v)), shape=(N, N))
    match = maximum_bipartite_matching(M, perm_type="column")
    return max(N - int((match >= 0).sum()), 1)


print("N = %d 个基因，L = %d 条调控边" % (N_real, len(u)))

In [ ]:
n_rand = 5
nd_real = None      # TODO: n_driver(u, v, N_real) / N_real
nd_rand = []        # TODO: n_rand 次保度随机化的 n_D
frac_source = None  # TODO: 源节点数 / N_D（源节点：入度为 0）

In [ ]:
# 核对（不用修改）
assert nd_real is not None and frac_source is not None, "nd_real、frac_source 还没有赋值"
assert 0 < nd_real <= 1, "n_D 是比例，应在 (0, 1] 之间；检查是否除以了 N_real"
assert isinstance(nd_rand, (list, tuple, np.ndarray)) and len(nd_rand) >= 3, "nd_rand 至少应有 3 次随机化的结果"
assert len(nd_rand) == n_rand, "nd_rand 的长度应等于 n_rand"
assert all(0 < x <= 1 for x in nd_rand), "每次随机化的 n_D 也应在 (0, 1] 之间"
assert 0 <= frac_source <= 1, "源节点一定是驱动节点，frac_source 应在 [0, 1] 之间；检查分母是否是驱动节点数"
print("✓ 任务 6 通过")